# YOLO11n 학습 -- YOLOv8s/YOLOv8n/YOLO26n과 동일 조건 4-way 비교용

`train_yolo_tile_augmented.ipynb`가 이미 만들어둔 증강 데이터셋
(`rummikub_data_3_tile_augmented`, 7:2:1 train/val/test, seed=42)을 그대로 재사용해서
**YOLO11n만** 추가로 학습한다. 데이터셋을 다시 만들지 않는다 -- 같은 `OUT_ROOT`
경로에 이미 있어야 하고, 없으면 이 노트북은 바로 실패한다(아래 셀 참고).

`EXPERIMENT_PLAN.md` 4.2절은 YOLO11n을 "시간 여유가 있을 때만 추가하는" 선택적
비교 대상으로 뒀다. mc/data 실제 홀드아웃 66장 검증(WORKLOG.md 29장)에서 YOLO26n이
증강 test 셋 mAP50은 YOLOv8s/YOLOv8n과 동일했음에도 실제 카메라에서는 PASS/FAIL
accuracy가 86.4%로 뚜렷이 낮았던 전례가 있다 -- 아키텍처 세대(YOLOv8 계열 73레이어
vs YOLO26 계열 122레이어)가 실제 원인일 수 있다는 가설이 있어서, YOLO11도 같은
방식으로 학습해 비교하면 그 가설을 더 검증할 수 있다. **단, YOLO26n 때와 마찬가지로
이 노트북의 증강 test 셋 결과만으로는 결론 내리지 말 것 -- 반드시 `analyze_holdout.py`
로 mc/data 실제 홀드아웃 재확인이 필요하다.**

학습 하이퍼파라미터(`YOLO_EPOCHS`, `YOLO_IMGSZ`, `batch=-1`, `seed`, `patience`,
`close_mosaic` 등)는 `train_yolo_tile_augmented.ipynb`/`train_yolo_tile_yolov8n.ipynb`와
값을 전부 동일하게 맞췄다.

cwd가 `.../boardgame-ai/rummikub_0730`라고 가정한 상대경로만 쓰며, 아직 실행하지 않은
상태다.

In [ ]:
from __future__ import annotations

import random
import shutil
from pathlib import Path

import torch

# train_yolo_tile_augmented.ipynb와 동일한 경로/시드 -- 새 데이터셋을 만들지 않고
# 그 노트북이 이미 만들어둔 결과물을 그대로 가리키기만 한다.
OUT_ROOT = Path("rummikub_data_3_tile_augmented")
WEIGHTS_OUT = Path("weights")
WEIGHTS_OUT.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. 데이터셋 재사용 확인

`train_yolo_tile_augmented.ipynb`를 먼저 실행해서 `dataset.yaml`이 이미 만들어져 있어야
한다. 없으면 여기서 바로 에러를 내고 멈춘다 -- 조용히 다른 데이터로 새로 만들면 기존
세 모델과 다른 데이터로 학습하게 돼서 비교 자체가 무효가 된다.

In [ ]:
dataset_yaml_path = OUT_ROOT / "dataset.yaml"
if not dataset_yaml_path.is_file():
    raise FileNotFoundError(
        f"{dataset_yaml_path.resolve()}가 없습니다. "
        "train_yolo_tile_augmented.ipynb의 1~6절(데이터셋 생성)을 먼저 실행하세요 -- "
        "여기서 데이터셋을 새로 만들면 기존 모델들과 다른 데이터가 되어 비교가 "
        "무효가 됩니다."
    )
print(dataset_yaml_path.read_text(encoding="utf-8"))

for split in ("train", "val", "test"):
    n_images = len(list((OUT_ROOT / "images" / split).glob("*.jpg")))
    print(f"[{split}] 이미지 {n_images}장")

## 2. 학습: YOLO11n

하이퍼파라미터는 `train_yolo_tile_augmented.ipynb`/`train_yolo_tile_yolov8n.ipynb`와
동일하다(`epochs=100`, `imgsz=960`, `batch=-1`, `patience=25`, `seed=42`,
`deterministic=True`, `close_mosaic=10`). `batch=-1`(자동 배치)은 모델 크기에 따라
실제 배치 크기가 달라질 수 있다는 점에 유의 -- `EXPERIMENT_PLAN.md` 4.2절 참고.

In [ ]:
from ultralytics import YOLO

YOLO_EPOCHS = 100
YOLO_IMGSZ = 960

yolo11n_model = YOLO("yolo11n.pt")
yolo11n_model.train(
    data=str(dataset_yaml_path.resolve()),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMGSZ,
    batch=-1,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    patience=25,
    project="runs/detect",
    name="rummikub_yolo11n_tile_augmented",
    pretrained=True,
    seed=RANDOM_SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
)

yolo11n_best_path = Path(yolo11n_model.trainer.best)
yolo11n_out_path = WEIGHTS_OUT / "yolo11n_tile_detector_best.pt"
shutil.copy2(yolo11n_best_path, yolo11n_out_path)
print("YOLO11n 가중치 저장:", yolo11n_out_path.resolve())

## 3. test 성능 4-way 비교

방금 학습한 YOLO11n과, 이미 `weights/`에 있는 YOLOv8s/YOLOv8n/YOLO26n 가중치를 같은
test split으로 평가해서 나란히 비교한다 (재학습하지 않고 저장된 가중치를 로드만 함).
가중치 파일이 이 노트북을 돌리는 머신의 `weights/`에 없으면 해당 모델 비교는 건너뛴다.

In [ ]:
def eval_on_test(model: YOLO, name: str):
    metrics = model.val(
        data=str(dataset_yaml_path.resolve()),
        split="test",
        imgsz=YOLO_IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        project="runs/detect",
        name=f"{name}_eval",
        plots=True,
    )
    print(
        f"{name:12s} P={metrics.box.mp:.3f} R={metrics.box.mr:.3f} "
        f"mAP50={metrics.box.map50:.3f} mAP50-95={metrics.box.map:.3f}"
    )


eval_on_test(yolo11n_model, "yolo11n")

other_weights = (
    ("yolov8n", "yolov8n_tile_detector_best.pt"),
    ("yolov8s", "yolov8s_tile_detector_best.pt"),
    ("yolo26n", "yolo26n_tile_detector_best.pt"),
)
for name, weight_name in other_weights:
    weight_path = WEIGHTS_OUT / weight_name
    if not weight_path.is_file():
        print(f"[skip] {weight_path.resolve()}가 없어 {name} 비교는 건너뜁니다.")
        continue
    eval_on_test(YOLO(str(weight_path)), name)

## 참고

- `weights/yolo11n_tile_detector_best.pt`가 만들어지면, `mc/pipeline_4.py`를 복제한
  `pipeline_5.py`(YOLO_WEIGHTS_PATH만 이 가중치로 교체) + `analyze_holdout.py`의
  `MODELS` dict에 조건부 추가하는 방식으로 mc/data 실제 홀드아웃 검증에 편입할 수
  있다 (아직 코드로 만들지 않음 -- 이 학습이 끝난 뒤 필요하면 요청).
- 이 노트북은 학습/평가만 담당한다. 여기서 나온 증강 test 셋 결과가 좋아도, YOLO26n
  사례처럼 실제 카메라에서는 다를 수 있으므로 최종 판단은 보류한다.